# Theta analysis

Calculate spike rhythmicity, LFP phase locking and an exploratory phase-precession candidate test. This workflow needs saved tracking/spikes and a saved LFP reference, independently of any CB/CD analysis outputs. It shares the existing movement and spatial-coverage selection rules.

Run on `hhw9l84` with the `rfmapping` kernel. Save results here, then open `theta_plotting.ipynb` to draw them without recalculation.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from EBB.center_cell_io import file_evidence, load_center_session, load_motive_center_session, load_lfp
from EBB.center_cells import CenterCellConfig, behavior_intervals, prepare_center_behavior, select_moving_spikes
from Utils.theta_analysis import analyze_theta_units, extract_theta_phase, summarize_lfp_reference
from Utils.theta_io import save_theta_analysis


## Recording and LFP reference


In [ ]:
recording_root = Path("/mnt/senzailab/Kai/#Recording/m20")
date = "260918"
recording_number = 11
probe_name = "A"
phase_key = "baseline"
is_motive = False  # Explicit camera choice: True = Motive circle; False = Basler rectangle.

session_id = f"{recording_root.name}_{date}_{recording_number}"
session_dir = recording_root / date / f"{date}_{recording_number}"
config = CenterCellConfig()
lfp_channel = 0  # Zero-based probe channel; phase zero is its positive peak.
lfp_path = session_dir / "data" / "lfp" / f"Probe{probe_name}" / phase_key / f"channel{lfp_channel}.npz"
output_dir = session_dir / "data" / "theta" / f"Probe{probe_name}" / phase_key / f"channel{lfp_channel}"
overwrite = False


## Load saved inputs and select movement


In [ ]:
loader = load_motive_center_session if is_motive else load_center_session
session_data = loader(session_dir, probe=probe_name, phase=phase_key)
behavior = prepare_center_behavior(
    session_data["times_s"], session_data["xy_cm_y_down"],
    session_data["hd_cw_deg"], session_data["arena_size_cm"],
    valid_mask=session_data["valid_mask"],
    arena_shape="circle" if is_motive else "rectangle", config=config,
)
session_summary = pd.Series({
    "good_units": len(session_data["spikes_by_unit"]),
    "duration_s": np.diff(behavior["trial_interval_s"])[0],
    "moving_time_s": behavior["exposure_s"][behavior["moving_mask"]].sum(),
    "spatial_coverage": behavior["spatial_coverage"],
    "excluded_tracking_fraction": session_data["tracking_selection"]["excluded_fraction"],
}, name=session_id)
display(session_summary)


In [ ]:
lfp = load_lfp(lfp_path)
theta_phase = extract_theta_phase(
    lfp["times_s"], lfp["voltage_uv"], sample_rate_hz=lfp["sample_rate_hz"],
)
moving_intervals = behavior_intervals(behavior, behavior["moving_mask"])
tracking_intervals = behavior_intervals(behavior, behavior["valid_position_mask"])
moving_spikes = {
    unit: select_moving_spikes(behavior, spikes)
    for unit, spikes in session_data["spikes_by_unit"].items()
}
lfp_reference = summarize_lfp_reference(lfp, theta_phase, moving_intervals)
lfp_metadata = lfp["metadata"]
del lfp


## Calculate theta measurements

Rhythmicity uses a 4–11 Hz ACG spectral peak and TRI ≥5. Phase locking reports MVL, Rayleigh p and Benjamini–Hochberg q; its saved classification still uses raw p <0.01. The phase-precession test checks spike rhythmicity relative to the LFP and is not a position–phase regression. ACG pairs stay inside continuous tracking/LFP intervals. Low-spike or insufficient-coverage units retain NA classifications.


In [ ]:
theta_results = analyze_theta_units(
    moving_spikes, theta_phase,
    tracking_intervals_s=tracking_intervals, moving_intervals_s=moving_intervals,
    min_spikes_exclusive=config.min_spikes_exclusive,
    spatial_coverage=behavior["spatial_coverage"],
    min_spatial_coverage_exclusive=config.min_spatial_coverage_exclusive,
)
theta_results["metadata"].update(reference_channel=lfp_channel, lfp_source=lfp_metadata)
display(theta_results["summary"])


## Save CSV and plot data

`theta_cells.csv` contains per-unit statistics, statuses, nullable classifications and q values. The companion NPZ and manifest preserve ACGs, spectra, phase counts/occupancy, all parameters and the LFP reference trace. Plotting only needs this saved directory.


In [ ]:
geometry = {
    "arena_shape": behavior["arena_shape"],
    "arena_size_cm": behavior["arena_size_cm"].tolist(),
}
if is_motive:
    geometry.update(session_data["geometry"])
provenance = {
    "session_id": session_id, "probe": probe_name, "phase": phase_key,
    "is_motive": is_motive, "input_files": session_data["provenance"],
    "geometry": geometry, "tracking_selection": session_data["tracking_selection"],
    "behavior_metadata": behavior["metadata"],
}


In [ ]:
provenance["lfp_file"] = file_evidence(lfp_path)
save_theta_analysis(
    output_dir, theta_results, provenance=provenance,
    lfp_reference=lfp_reference, overwrite=overwrite,
)
session_summary.to_csv(output_dir / "session_qc.csv")
print(f"Saved theta results: {output_dir}")


## Explicit first-time LFP generation

If the LFP cache is missing, run this separately, then rerun the loading cell. Loading never generates a cache.

```python
from EBB.center_cell_io import generate_lfp_from_open_ephys, save_lfp

lfp = generate_lfp_from_open_ephys(
    session_dir, probe=probe_name, channel=lfp_channel,
    interval_s=behavior["trial_interval_s"],
    adc_time_origin_s=session_data["adc_time_origin_s"],
)
save_lfp(lfp_path, lfp)
del lfp
```
